# EDA — Credit Card Fraud Dataset

Appendix notebook. Detailed charts live here rather than in the main README, which stays focused on
architecture, latency, and cost curves. Covers: class imbalance ratio, amount/time distribution by class,
feature diagnostics, and correlation structure.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

df = pd.read_csv("../data/raw/creditcard.csv")
df.shape

## Class distribution and imbalance ratio

In [ ]:
counts = df["Class"].value_counts()
ratio = counts[0] / counts[1]
print(counts)
print(f"imbalance ratio (legit:fraud) = {ratio:.1f}:1")
counts.plot(kind="bar", title="Class counts (0=legit, 1=fraud)")

## Amount / Time distribution by class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Class", y="Amount", ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Amount by class (log scale)")
sns.histplot(data=df, x="Time", hue="Class", bins=48, stat="density", common_norm=False, ax=axes[1])
axes[1].set_title("Transaction time distribution by class")
fig.tight_layout()

## Feature diagnostics (distributional, informing model design)

In [ ]:
v_cols = [c for c in df.columns if c.startswith("V")]
means_by_class = df.groupby("Class")[v_cols].mean().T
means_by_class["abs_diff"] = (means_by_class[0] - means_by_class[1]).abs()
means_by_class.sort_values("abs_diff", ascending=False).head(10)

The largest mean-shift components (commonly V14, V17 on this dataset) line up with what published analyses
of this dataset and the SHAP results in Day 5 highlight as the strongest fraud signal among the PCA
components.

## Correlation structure

In [ ]:
corr = df[v_cols + ["Amount", "Class"]].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature correlation matrix")